# Step 1: Load and Explore the Dataset

The e-commerce dataset is loaded into a Spark DataFrame. 
Basic exploration is performed to view records, columns, schema, and total number of rows.

In [8]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 5") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.1.2


In [9]:
df = spark.read.csv(
    "../data/ecommerce_dataset.csv",
    header=True,
    inferSchema=True
)

print("Dataset loaded successfully!")

Dataset loaded successfully!


In [10]:
df.show(5)

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+--------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|            email|username|   price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+-----------------+--------+--------+--------+--------+
|  18032|      2025-12-14|  West|         Grocery|    7018.55|  Pending|   Mumbai| 42|    Standard|2025-12-21 19:34:33|user0@example.com|  user_0|12681.93|      75|       6|
|  79656|      2026-03-07| South|          Sports|   39489.83|Cancelled|   Mumbai| 24|       Basic|2025-08-15 20:21:53|user1@example.com|  user_1| 12538.3|      42|       2|
|  68911|      2026-06-28| South|          Sports|   24189.61|Cancelled|  Chennai| 63|       Basic|2025-06-20 13:11:41|user2@examp

In [11]:
print(df.columns)

['user_id', 'transaction_date', 'region', 'product_category', 'sale_amount', 'status', 'city', 'age', 'subscription', 'raw_timestamp', 'email', 'username', 'price', 'store_id', 'quantity']


In [12]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- price: double (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)



In [7]:
print('Total Records : ', df.count())

Total Records :  300000


## Step 2: Data Cleaning

In this step, duplicate records, null values, empty values, and inconsistent data are identified and cleaned using PySpark DataFrame operations.

In [13]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total Rows:", total_rows)
print("Unique Rows:", unique_rows)
print("Duplicate Rows:", total_rows - unique_rows)

Total Rows: 300000
Unique Rows: 295000
Duplicate Rows: 5000


In [14]:
df_clean = df.dropDuplicates()
print('Rows after removing duplicates : ' , df_clean.count())

Rows after removing duplicates :  295000


In [15]:
from pyspark.sql.functions import col, sum

df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_clean.columns
]).show()

+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|status|city|age|subscription|raw_timestamp|email|username|price|store_id|quantity|
+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+
|      0|               0|     0|               0|       3000|  3000|   0|  0|           0|            0| 4981|    4978| 3000|       0|       0|
+-------+----------------+------+----------------+-----------+------+----+---+------------+-------------+-----+--------+-----+--------+--------+



In [16]:
df_clean = df_clean.fillna({
    "price": 0,
    "sale_amount": 0
})

df_clean = df_clean.fillna({
    "status": "Unknown",
    "email": "Unknown",
    "username": "Unknown"
})

In [17]:
from pyspark.sql.functions import when, col

df_clean = df_clean.withColumn(
    "email",
    when(col("email") == "", "Unknown").otherwise(col("email"))
)

df_clean = df_clean.withColumn(
    "username",
    when(col("username") == "", "Unknown").otherwise(col("username"))
)

In [18]:
df_clean.filter(
    col("raw_timestamp") == "invalid_timestamp"
).show(5)

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-----------------+--------------------+----------+--------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|    raw_timestamp|               email|  username|   price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-----------------+--------------------+----------+--------+--------+--------+
|  90626|      2025-08-17|  East|         Grocery|    12045.5|Cancelled|Hyderabad| 40|     Premium|invalid_timestamp|user36139@example...|user_36139|10345.19|     251|       4|
|  34633|      2026-01-25|  West|     Electronics|   26628.75|Cancelled|    Surat| 22|    Standard|invalid_timestamp|user25254@example...|user_25254| 11696.5|     240|       2|
|  46365|      2025-10-01| North|        Clothing|   24944.24|  Pending|Hyderabad| 22|       Basic|invalid_timestam

In [19]:
df_clean = df_clean.filter(
    col("raw_timestamp") != "invalid_timestamp"
)

In [20]:
print("Final rows after cleaning:", df_clean.count())

Final rows after cleaning: 293500


### Step 2 Insights

- The dataset initially contained 300,000 records.
- 5,000 duplicate records were identified and removed.
- Null values were found in sale_amount, status, email, username, and price columns.
- Numeric null values were replaced with 0, while categorical null values were replaced with "Unknown".
- Empty email and username values were handled.
- Invalid timestamp records were identified and removed.
- After data cleaning, 293,500 valid records remained.

## Step 3: Filter Data

In this step, filtering conditions are applied to the cleaned dataset based on age, product category, and region.

In [21]:
age_filtered = df_clean.filter(col('age').between(18,30))
age_filtered.show(5)
print('Records with age between 18 to 30 : ', age_filtered.count())

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+-------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email| username|  price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+-------+--------+--------+
|  45782|      2026-04-02|  West|          Sports|   40498.81|  Pending|  Chennai| 27|     Premium|2025-09-18 13:06:56| user439@example.com| user_439|3983.73|     352|       5|
|  36001|      2025-09-08| South|           Books|   32327.99|Cancelled|    Delhi| 23|       Basic|2025-01-21 21:00:11| user840@example.com| user_840|2780.07|     347|       9|
|  78368|      2026-06-12|  East|         Grocery|   45123.11|Cancelled|    Surat| 19|    Standard|2025-05-12 20:34

In [22]:
cat_filtered = df_clean.filter(col('product_category') ==  'Electronics')
cat_filtered.show(5)
print('Electronics Records:',cat_filtered.count())

+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+--------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|     city|age|subscription|      raw_timestamp|               email| username|   price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+---------+---+------------+-------------------+--------------------+---------+--------+--------+--------+
|  46919|      2026-02-13| South|     Electronics|    1031.78|Completed|Hyderabad| 33|       Basic|2025-06-15 00:09:39| user372@example.com| user_372|14626.81|     373|       9|
|  19530|      2025-11-12| North|     Electronics|   26674.73|Completed|Bengaluru| 60|     Premium|2026-06-21 02:48:36| user459@example.com| user_459|  9146.0|      83|       2|
|  65771|      2025-03-06|  West|     Electronics|   20888.29|Cancelled|   Mumbai| 38|       Basic|2025-03-30 

In [23]:
region_filtered = df_clean.filter(
    col("region") == "West"
)
region_filtered.show(5)
print("West Region Records:", region_filtered.count())

+-------+----------------+------+----------------+-----------+---------+-------+---+------------+-------------------+--------------------+---------+--------+--------+--------+
|user_id|transaction_date|region|product_category|sale_amount|   status|   city|age|subscription|      raw_timestamp|               email| username|   price|store_id|quantity|
+-------+----------------+------+----------------+-----------+---------+-------+---+------------+-------------------+--------------------+---------+--------+--------+--------+
|  45782|      2026-04-02|  West|          Sports|   40498.81|  Pending|Chennai| 27|     Premium|2025-09-18 13:06:56| user439@example.com| user_439| 3983.73|     352|       5|
|  65771|      2025-03-06|  West|     Electronics|   20888.29|Cancelled| Mumbai| 38|       Basic|2025-03-30 16:12:25|user1660@example.com|user_1660| 2145.96|     167|      10|
|  90123|      2025-04-24|  West|     Electronics|   37790.33|Completed|  Surat| 29|     Premium|2025-12-27 03:50:25|use

In [24]:
premium_users = df_clean.filter(
    (col("age").between(18, 30)) &
    (col("subscription") == "Premium")
)
premium_users.select(
    "user_id",
    "age",
    "subscription"
).show(10)

+-------+---+------------+
|user_id|age|subscription|
+-------+---+------------+
|  45782| 27|     Premium|
|  18047| 28|     Premium|
|  90123| 29|     Premium|
|  77093| 21|     Premium|
|  67440| 23|     Premium|
|  35491| 21|     Premium|
|  65604| 24|     Premium|
|  54002| 28|     Premium|
|  80502| 23|     Premium|
|  80828| 29|     Premium|
+-------+---+------------+
only showing top 10 rows


### Step 3 Insights

- Records were filtered using age, product category, and region.
- The between() function was used to select users aged between 18 and 30.
- Electronics transactions and West region records were filtered separately.
- Multiple conditions were combined to identify Premium users aged between 18 and 30.
- Spark filter() returns a new DataFrame because Spark DataFrames are immutable.

## Step 4: Data Transformation and Schema Modification

In this step, columns are renamed, data types are changed, and new derived columns are created using PySpark transformations.

In [25]:
df_transformed = df_clean.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

df_transformed.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = false)
 |-- status: string (nullable = false)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- email: string (nullable = false)
 |-- username: string (nullable = false)
 |-- price: double (nullable = false)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)



In [26]:
from pyspark.sql.types import TimestampType

df_transformed = df_transformed.withColumn(
    "event_time",
    col("event_time").cast(TimestampType())
)

df_transformed.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = false)
 |-- status: string (nullable = false)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = false)
 |-- username: string (nullable = false)
 |-- price: double (nullable = false)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)



In [27]:
df_transformed = df_transformed.withColumn(
    "total_amount",
    col("price") * col("quantity")
)

df_transformed.select(
    "price",
    "quantity",
    "total_amount"
).show(10)

+--------+--------+------------------+
|   price|quantity|      total_amount|
+--------+--------+------------------+
| 5428.97|       8|          43431.76|
|14626.81|       9|         131641.29|
| 3983.73|       5|          19918.65|
|  9146.0|       2|           18292.0|
| 7521.59|       3|          22564.77|
| 2780.07|       9|          25020.63|
| 4409.65|       6|26457.899999999998|
| 1352.53|       3|           4057.59|
| 4155.27|       3|12465.810000000001|
| 8122.34|       8|          64978.72|
+--------+--------+------------------+
only showing top 10 rows


In [28]:
df_transformed.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: double (nullable = false)
 |-- status: string (nullable = false)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- email: string (nullable = false)
 |-- username: string (nullable = false)
 |-- price: double (nullable = false)
 |-- store_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_amount: double (nullable = true)



### Step 4 Insights

- The raw_timestamp column was renamed to event_time.
- The event_time column was converted from string to TimestampType.
- A new derived column named total_amount was created using price and quantity.
- Spark transformations return new DataFrames because DataFrames are immutable.
- Schema modification ensures correct data types for further processing and analysis.

## Step 5: Aggregation and Basic Analysis

In this step, aggregation functions are used to calculate total records, total sales, average price, minimum price, and maximum price.

In [29]:
total_records = df_transformed.count()
print('Total Records : ' , total_records)

Total Records :  293500


In [32]:
from pyspark.sql.functions import sum

df_transformed.select(
    sum("sale_amount").alias("Total Sales")
).show()

+-------------------+
|        Total Sales|
+-------------------+
|7.292793676980007E9|
+-------------------+



In [33]:
from pyspark.sql.functions import avg

df_transformed.select(
    avg("price").alias("Average Price")
).show()

+----------------+
|   Average Price|
+----------------+
|9921.66471100512|
+----------------+



In [34]:
from pyspark.sql.functions import min, max

df_transformed.select(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price")
).show()

+-------------+-------------+
|Minimum Price|Maximum Price|
+-------------+-------------+
|          0.0|     19999.99|
+-------------+-------------+



In [35]:
from pyspark.sql.functions import min, max, avg

df_transformed.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+----------------+
|Minimum Price|Maximum Price|   Average Price|
+-------------+-------------+----------------+
|          0.0|     19999.99|9921.66471100512|
+-------------+-------------+----------------+



### Step 5 Insights

- The total number of records was calculated using count().
- The sum() function was used to calculate total sales.
- The avg() function calculated the average product price.
- Minimum and maximum prices were identified using min() and max().
- The agg() function was used to calculate multiple summary statistics in a single operation.
- Aggregation functions provide useful statistical insights from large datasets.

## Step 6: Grouping and Aggregated Analysis

In this step, data is grouped using groupBy() and aggregate functions such as count(), sum(), and avg() are applied to analyze grouped records.

In [36]:
city_count = df_transformed.groupBy('city').count()
city_count.show()

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|37012|
|   Mumbai|36589|
|    Surat|36490|
|     Pune|36747|
|    Delhi|36819|
|Bengaluru|36603|
|Hyderabad|36474|
|   Jaipur|36766|
+---------+-----+



In [37]:
city_count.filter(
    col('count') > 100
).show()

+---------+-----+
|     city|count|
+---------+-----+
|  Chennai|37012|
|   Mumbai|36589|
|    Surat|36490|
|     Pune|36747|
|    Delhi|36819|
|Bengaluru|36603|
|Hyderabad|36474|
|   Jaipur|36766|
+---------+-----+



In [38]:
cat_avg = df_transformed.groupBy(
    'product_category'
).agg(
    avg('sale_amount').alias('Average sale amount')
)
cat_avg.show()

+----------------+-------------------+
|product_category|Average sale amount|
+----------------+-------------------+
|          Sports| 24722.980860172727|
|         Grocery| 24845.178594711946|
|     Electronics| 24881.461330898594|
|        Clothing| 24888.405362769012|
|           Books| 24899.935990684327|
+----------------+-------------------+



In [40]:
region_sales = df_transformed.groupBy(
    'region'
).agg(
    sum('sale_amount').alias('Total Sales')
)
region_sales.show()

+------+--------------------+
|region|         Total Sales|
+------+--------------------+
| South|     1.82147076579E9|
|  East|     1.82724119263E9|
|  West| 1.825023044810001E9|
| North|1.8190586737499988E9|
+------+--------------------+



In [42]:
regional_category_analysis = df_transformed.groupBy(
    "region",
    "product_category"
).agg(
    avg("sale_amount").alias("Average Sales"),
    sum("sale_amount").alias("Total Sales")
)

regional_category_analysis.show()

+------+----------------+------------------+--------------------+
|region|product_category|     Average Sales|         Total Sales|
+------+----------------+------------------+--------------------+
| South|         Grocery|24926.531957768337|3.6948598321000004E8|
| South|        Clothing| 24950.34072173381| 3.660963494100002E8|
| North|        Clothing|24905.602137248214| 3.647425433000001E8|
| North|     Electronics| 24608.28731555855| 3.645717765799999E8|
|  East|     Electronics|24770.099425583936| 3.648140243400002E8|
| South|          Sports|  24682.8596396272|      3.5753122188E8|
| South|           Books|24780.985866730392|3.6296709999000007E8|
|  East|        Clothing|24789.951597405256|3.6686649369000036E8|
|  West|        Clothing| 24908.88297706231|3.6053117220999986E8|
| North|         Grocery| 24703.45392905223|3.6699451156999993E8|
|  East|          Sports|24692.695413657937|3.5905648401000005E8|
|  East|           Books| 25118.48209487769| 3.717032980400001E8|
|  West|  

### Step 6 Insights

- Data was grouped using the groupBy() function.
- The total number of records for each city was calculated.
- Aggregated results were filtered based on count values.
- Average sales were calculated for each product category.
- Total sales were calculated for each region.
- Multiple columns were used for grouped analysis.
- groupBy() may cause a shuffle because records with the same key must be moved across partitions.
- Therefore, groupBy() is considered a wide transformation.

## Step 7: Complete Data Processing Pipeline

This step combines data cleaning, filtering, transformation, and aggregation into a complete PySpark data processing pipeline.

In [43]:
from pyspark.sql.functions import col, sum

final_result = (
    df
    .dropDuplicates()
    .fillna({
        "price": 0,
        "sale_amount": 0,
        "status": "Unknown",
        "email": "Unknown",
        "username": "Unknown"
    })
    .filter(col("raw_timestamp") != "invalid_timestamp")
    .filter(col("region") == "West")
    .withColumn(
        "total_revenue",
        col("price") * col("quantity")
    )
    .groupBy("store_id")
    .agg(
        sum("total_revenue").alias("total_revenue")
    )
)

final_result.show(10)

+--------+-----------------+
|store_id|    total_revenue|
+--------+-----------------+
|     463|       8258585.75|
|     496|       7439896.61|
|     471|        7887960.1|
|     148|       8504793.18|
|     243|       6622744.86|
|     392|       7233751.67|
|      31|8826700.169999998|
|      85|       8434576.18|
|     137|       8690009.23|
|     451|8078350.509999999|
+--------+-----------------+
only showing top 10 rows


In [44]:
final_result.orderBy(
    col("total_revenue").desc()
).show(10)

+--------+--------------------+
|store_id|       total_revenue|
+--------+--------------------+
|     424|       1.056583154E7|
|     281|       1.047746734E7|
|     227|1.0365739889999999E7|
|     483|1.0201577559999999E7|
|      56|          9975349.53|
|      33|          9960960.15|
|     328|          9849693.69|
|      39|          9768104.65|
|      78|   9759966.479999999|
|     442|   9716725.979999999|
+--------+--------------------+
only showing top 10 rows


In [45]:
final_result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[store_id#143], functions=[sum(total_revenue#1467)])
   +- Exchange hashpartitioning(store_id#143, 200), ENSURE_REQUIREMENTS, [plan_id=2953]
      +- HashAggregate(keys=[store_id#143], functions=[partial_sum(total_revenue#1467)])
         +- HashAggregate(keys=[city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137, user_id#130, sale_amount#134, status#135, region#132, transaction_date#131, product_category#133, raw_timestamp#139, price#142], functions=[])
            +- Exchange hashpartitioning(city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137, user_id#130, sale_amount#134, status#135, region#132, transaction_date#131, product_category#133, raw_timestamp#139, price#142, 200), ENSURE_REQUIREMENTS, [plan_id=2949]
               +- HashAggregate(keys=[city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137

### Step 7 Insights

- Duplicate records were removed from the dataset.
- Missing numeric and categorical values were handled.
- Invalid timestamp records were filtered.
- Records from the West region were selected.
- A derived total_revenue column was created using price and quantity.
- Data was grouped by store_id to calculate total revenue for each store.
- The complete pipeline combines multiple PySpark transformations.
- Spark uses lazy evaluation and executes the transformations when an action is called.

## Step 8: Wide Transformations and Shuffle

This step demonstrates the concept of wide transformations and shuffle operations in Spark using groupBy() and repartition().

In [46]:
print(
    "Number of Partitions:",
    df_transformed.rdd.getNumPartitions()
)

Number of Partitions: 8


In [47]:
shuffle_result = (
    df_transformed
    .groupBy("region")
    .agg(
        sum("sale_amount").alias("Total Sales")
    )
)

shuffle_result.show()

+------+--------------------+
|region|         Total Sales|
+------+--------------------+
| South|     1.82147076579E9|
|  East|     1.82724119263E9|
|  West| 1.825023044810001E9|
| North|1.8190586737499988E9|
+------+--------------------+



In [48]:
shuffle_result.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[region#132], functions=[sum(sale_amount#460)])
   +- Exchange hashpartitioning(region#132, 200), ENSURE_REQUIREMENTS, [plan_id=3163]
      +- HashAggregate(keys=[region#132], functions=[partial_sum(sale_amount#460)])
         +- HashAggregate(keys=[city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137, user_id#130, sale_amount#134, status#135, region#132, transaction_date#131, product_category#133, raw_timestamp#139, price#142], functions=[])
            +- Exchange hashpartitioning(city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137, user_id#130, sale_amount#134, status#135, region#132, transaction_date#131, product_category#133, raw_timestamp#139, price#142, 200), ENSURE_REQUIREMENTS, [plan_id=3159]
               +- HashAggregate(keys=[city#136, quantity#144, email#140, username#141, store_id#143, subscription#138, age#137, user_id#13

In [49]:
repartitioned_df = df_transformed.repartition(
    4,
    "region"
)

print(
    "Partitions after repartition:",
    repartitioned_df.rdd.getNumPartitions()
)

Partitions after repartition: 4


### Step 8 Insights

- Spark divides data into partitions for parallel processing.
- groupBy() is a wide transformation because data with the same key may need to move across partitions.
- This movement of data between partitions is called shuffle.
- Shuffle can be expensive because it involves data redistribution.
- The execution plan can be viewed using explain().
- repartition() redistributes data and can cause a shuffle.

## Step 9: Save Processed Results

In this step, the final aggregated results are saved as CSV files for further analysis and reporting.

In [51]:
result_pandas = final_result.toPandas()

result_pandas.to_csv(
    "../Output/store_revenue_results.csv",
    index=False
)

print("Results saved successfully!")

Results saved successfully!


# Step 10: Final Pipeline Summary and Insights

The complete Spark data processing pipeline was successfully implemented.

## Processing Steps Performed

1. Loaded the CSV dataset into a Spark DataFrame.
2. Inspected columns, schema, and total records.
3. Removed duplicate records.
4. Handled null and empty values.
5. Removed inconsistent timestamp records.
6. Applied filtering based on age, region, category, and subscription.
7. Renamed columns and modified data types.
8. Performed aggregation using count, sum, average, minimum, and maximum.
9. Used groupBy operations to analyze grouped data.
10. Calculated total revenue for each store.
11. Saved the final aggregated results as a CSV file.

## Key Insights

- Spark DataFrames are immutable, so transformations return new DataFrames.
- Data cleaning improves the accuracy of aggregation results.
- groupBy operations may cause shuffle because data moves across partitions.
- Wide transformations can be more expensive due to network data movement.
- Spark provides efficient processing for large datasets using distributed computing.
- The final pipeline successfully combined data cleaning, transformation, filtering, and aggregation.